## Specify folder names and files to process

In [1]:
import numpy as np

# For folder names and simulation file structure refer to </README.md>
folder_name = r"F:\Contents\2024-08-20 Array noise" # The folder where all CST simulation files are located

# You can change `ch_index` and `size_index` here to choose between folders
ch_index = 0 # 0: 8-ch, 1: 12-ch, 2: 16-ch
size_index = 1 # 0: outer_radius=134.00, 1: outer_radius=171.28, 2: outer_radius=208.56
m_xy = np.array([1, 1j, 0]) / np.sqrt(2) # rotation; here we choose [1, j, 0]
m_mark = "1,j"  # a string to attach onto pictures

## Run

In [2]:
import itertools
import skrf
from rx_array_snr_evaluation import field
from importlib import reload
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from matplotlib import pyplot as plt

reload(field)

# There are nine simulation files
# Generates all folder names containing simulation results
# 12ch, outer_radius=134.00, loop_xy=97.60
# 12ch, outer_radius=171.28, loop_xy=120.57
# 12ch, outer_radius=208.56, loop_xy=143.93
#16ch, outer_radius=134.00, loop_xy=73.73
# 16ch, outer_radius=171.28, loop_xy=91.43
# 16ch, outer_radius=208.56, loop_xy=109.02
# 8ch, outer_radius=134.00, loop_xy=146.11
# 8ch, outer_radius=171.28, loop_xy=181.24
# 8ch, outer_radius=208.56, loop_xy=213.48

ch_n = np.array([8, 12, 16], dtype=np.int_)
sizes = np.array([134.00, 171.28, 208.56], dtype=np.float_)
loop_xys = np.array([[146.11, 181.24, 213.48], 
                     [97.60, 120.57, 143.93], 
                     [73.73, 91.43, 109.02]])
subfolder_names = [[], [], []]
for iter_m, iter_n in itertools.product([0, 1, 2], [0, 1, 2]):
    subfolder_name = f"{folder_name:s}\\{ch_n[iter_m]:d}ch, outer_radius={sizes[iter_n]:.2f}, loop_xy={loop_xys[iter_m, iter_n]:.2f}"
    subfolder_names[iter_m].append(subfolder_name)

def mask_ROI(position: np.ndarray, h_field: np.ndarray) -> np.ndarray: 
    """ Defines region of interest
        Now it's a circle centered at (0, 0), radius 100.
    """
    return np.linalg.norm(position[:, 0:2], axis=1) <= 100.

In [3]:
# Load `signal_generator_data` by `convert_hdf5_data.ipynb`
signal_vector_generator = field.SignalVectorGenerator.load(subfolder_names[ch_index][size_index] + "\\signal_generator_data")
print(subfolder_names[ch_index][size_index])
snp_net = skrf.Network(subfolder_names[ch_index][size_index] + f"\\matZ_{ch_n[ch_index]}_size{size_index+1}.s{ch_n[ch_index]}p") # scikit-rf processes the snp file
array_Z = snp_net.z[0, ...]
array_pos_roi0 = signal_vector_generator._position  # 2D grid

F:\Contents\2024-08-20 Array noise\8ch, outer_radius=171.28, loop_xy=181.24


In [4]:
snp_net.name

'matZ_8_size2'

In [5]:
# c = 4 * 1.380649E-23 * 290 * 50e3 # Set the bandwidth 50 kHz: https://mriquestions.com/receiver-bandwidth.html
c = 1 # It is "legal" to set c as 1--will be a common factor in (39)

# BFP740 3 V, 6 mA; BFR35AP 10 V, 10 mA
# Transistors: Noiseless, Elcry2-u (elcry.com), Infineon BFP740, Infineon BFR35AP
# Transistor noise parameters: F_{opt}—minimum noise figure (linear scale), Z_{opt}—optimal noise impedance, R_{no}—noise resistance, just a parameter
# Some transistor parameters are not listed in datasheets; they are extracted from ADS simulation. Just accept them as they are. 
transistor_names = ["Noiseless", "Elcry2-u", "BFP740", "BFR35AP"]
f_opt = 10 ** (np.array([0, 0.406208, 0.43869, 1.11818]) / 10)  
z_opt = np.array([50, 57.306557 + 21.569136j, 123.67115 + 3.12944j, 62.49675 + 2.55972j])
r_no = np.array([0, 1.926394, 6.15520, 8.01097], dtype=np.float_)

# Test piece
# transistor_names = ["ideal", "BFP740"]
# f_opt = 10 ** (np.array([0., 0.43869]) / 10)
# z_opt = np.array([123.67115 + 3.12944j, 123.67115 + 3.12944j])
# r_no = np.array([0., 6.15520], dtype=np.float_)
g_noise = (r_no / np.abs(z_opt) ** 2).astype(np.float_)  # Rn / z_opt ** 2, equivalent to matrix [D]
lange_N = np.real(z_opt) * g_noise

center_location_all, _ = field.SNRMapper.find_nearest_point(array_pos_roi0, np.array([0, 0, 115]))  # Find the index of point [0, 0, 115]
print(f"{center_location_all=}")
us = signal_vector_generator.signal_vectors(m_xy, normalize=True, normalize_to_pos=center_location_all)

center_location_all=392672


In [6]:
np.abs(us[..., center_location_all])
np.diff(np.angle(us[..., center_location_all], True))

array([0.34613505, 0.3594169 , 0.34691701, 0.36177693, 0.34639871,
       0.35848153, 0.34703951, 0.36171614])

array([314.85760425, -44.9521795 , -45.00825376, -45.04023023,
       -45.00357466, -44.9077924 , -45.14879491])

In [7]:
lange_N

array([0.        , 0.02944441, 0.04973885, 0.12796752])

In [8]:
us.shape

(8, 785343)

In [9]:

# This part needs to be made into a list
z_generators = []
noise_match_cores = []
signal_opt_cores = []
admissibilities = []
snr_mappers = []
for iter_n in range(len(f_opt)):
    z_generator = field.ImpedanceMatrixGenerator(array_Z, c, f_opt[iter_n], lange_N[iter_n])
    noise_match_core = z_generator.noise_match_core()
    
    signal_opt_core, us_admissibility = z_generator.signal_optimization_core(us)
    print(f"{transistor_names[iter_n]}, Admissible proportion = {np.sum(us_admissibility) / np.size(us_admissibility):%}")

    # signal_center_core = signal_opt_core[center_location_filtered, ...]#[np.newaxis, ...]
    snr_mapper = field.SNRMapper(us, array_pos_roi0, noise_match_core, 
                                second_ref_core_39=z_generator._inv_A)
    z_generators.append(z_generator)
    admissibilities.append(us_admissibility)
    signal_opt_cores.append(signal_opt_core)
    snr_mappers.append(snr_mapper)
    noise_match_cores.append(noise_match_core)

    

Noiseless, Admissible proportion = 5.108214%
Elcry2-u, Admissible proportion = 5.108214%
BFP740, Admissible proportion = 5.108214%
BFR35AP, Admissible proportion = 5.108214%


In [10]:
array_pos_roi0_admissible = array_pos_roi0[us_admissibility, ...]  # Discards inadmissible regions
# If you discard inadmissible regions, the array itself will change, namely it will shorten. In this case, the index of image center [0, 0, 115] can shift, for which you'll need to find `center_location_filtered`.


# center_location_filtered = find_nearest_position(np.sum((array_pos_roi0_admissible - np.array([[0, 0, 115]])) ** 2, axis=1), 0)
# center_location_all = find_nearest_position(np.sum((array_pos_roi0 - np.array([[0, 0, 115]])) ** 2, axis=1), 0)
if np.size(array_pos_roi0_admissible) > 0:
    center_location_filtered, _ = field.SNRMapper.find_nearest_point(array_pos_roi0_admissible, np.array([0, 0, 115]))
    print(f"{center_location_filtered=}")
    print(array_pos_roi0_admissible[center_location_filtered, ...])
else:
    print("Inadmissible")


center_location_filtered=21759
[  0.   0. 115.]


In [11]:
for iter_n in range(len(f_opt)):
    if np.size(array_pos_roi0_admissible) > 0:
        snr_mappers[iter_n].core_39 = signal_opt_cores[iter_n][center_location_filtered, ...]
    z_generators[iter_n].save(subfolder_names[ch_index][size_index] + f"\\z_matrix_generator_data-{transistor_names[iter_n]:s}-[{m_mark}]")
    snr_mappers[iter_n].save(subfolder_names[ch_index][size_index] + f"\\snr_mapper_data-{transistor_names[iter_n]:s}-[{m_mark}]")

In [12]:
for iter_n in range(len(f_opt)):
    print(signal_opt_cores[iter_n].shape)

(40117, 8, 8)
(40117, 8, 8)
(40117, 8, 8)
(40117, 8, 8)
